In [1]:
!pip install langchain-text-splitters langchain_experimental spacy tiktoken sentence-transformers
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
TEXT = '''Artificial intelligence is transforming healthcare rapidly. Machine learning models can now detect desease from medical images with remarkable accuracy. This technology promises to make diagnosis faster and more accessible worldwide.'''

In [6]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter, SpacyTextSplitter

In [7]:
# Paragraph Splitter
paragaraph_splitter = CharacterTextSplitter()
para = paragaraph_splitter.split_text(TEXT)
para

['Artificial intelligence is transforming healthcare rapidly. Machine learning models can now detect desease from medical images with remarkable accuracy. This technology promises to make diagnosis faster and more accessible worldwide.']

In [8]:
# Sentence Splitter
sentence_splitter = SpacyTextSplitter(pipeline="en_core_web_sm")
sent = sentence_splitter.split_text(TEXT)
sent

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


['Artificial intelligence is transforming healthcare rapidly.\n\nMachine learning models can now detect desease from medical images with remarkable accuracy.\n\nThis technology promises to make diagnosis faster and more accessible worldwide.']

In [12]:
from ast import fix_missing_locations
# Fixed size chunking
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
fixed = fixed_splitter.split_text(TEXT)
fixed

['Artificial intelligence is transforming healthcare rapidly. Machine learning models can now detect',
 'can now detect desease from medical images with remarkable accuracy. This technology promises to',
 'promises to make diagnosis faster and more accessible worldwide.']

In [13]:
# Sliding Window generally we use RecursiveCharacterTextSplitter as Token splitter takes the whole word token rather than character
sliding_splitter = TokenTextSplitter(chunk_size=20, chunk_overlap=10, encoding_name='cl100k_base')
sliding = sliding_splitter.split_text(TEXT)
sliding

['Artificial intelligence is transforming healthcare rapidly. Machine learning models can now detect desease from medical images with',
 ' models can now detect desease from medical images with remarkable accuracy. This technology promises to make diagnosis faster',
 ' remarkable accuracy. This technology promises to make diagnosis faster and more accessible worldwide.']

In [15]:
# Semantic Chunking
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
hf = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
semantic_splitter = SemanticChunker(hf)
semantic_chunks = semantic_splitter.split_text(TEXT)
semantic_chunks

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

['Artificial intelligence is transforming healthcare rapidly. Machine learning models can now detect desease from medical images with remarkable accuracy.',
 'This technology promises to make diagnosis faster and more accessible worldwide.']

In [20]:
# Entity based Chunking
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("I am Syed Askari. I live in UK and work for an American bank")
sent_texts = [s.text.strip() for s in doc.sents if s.text.strip()]

print(sent_texts)

entity_map = {}
for ent in doc.ents:
  entity_map.setdefault(ent.text, set())
print(entity_map)

for s in sent_texts:
  s_doc = nlp(s)

  for ent in s_doc.ents:
    entity_map.setdefault(ent.text, set()).add(s)

entity_chunks = []
for ent, ss in entity_map.items():
  entity_chunks.append(f"ENTITY : {ent}\n" + "\n".join(f"-{X}" for X in sorted(ss)))

entity_chunks

['I am Syed Askari.', 'I live in UK and work for an American bank']
{'Syed Askari': set(), 'UK': set(), 'American': set()}


['ENTITY : Syed Askari\n-I am Syed Askari.',
 'ENTITY : UK\n-I live in UK and work for an American bank',
 'ENTITY : American\n-I live in UK and work for an American bank']

In [23]:
# How to save Metadata information

from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=10, chunk_overlap=0, separator='', strip_whitespace=False)

texts = text_splitter.create_documents([TEXT])
texts

[Document(metadata={}, page_content='Artificial'),
 Document(metadata={}, page_content=' intellige'),
 Document(metadata={}, page_content='nce is tra'),
 Document(metadata={}, page_content='nsforming '),
 Document(metadata={}, page_content='healthcare'),
 Document(metadata={}, page_content=' rapidly. '),
 Document(metadata={}, page_content='Machine le'),
 Document(metadata={}, page_content='arning mod'),
 Document(metadata={}, page_content='els can no'),
 Document(metadata={}, page_content='w detect d'),
 Document(metadata={}, page_content='esease fro'),
 Document(metadata={}, page_content='m medical '),
 Document(metadata={}, page_content='images wit'),
 Document(metadata={}, page_content='h remarkab'),
 Document(metadata={}, page_content='le accurac'),
 Document(metadata={}, page_content='y. This te'),
 Document(metadata={}, page_content='chnology p'),
 Document(metadata={}, page_content='romises to'),
 Document(metadata={}, page_content=' make diag'),
 Document(metadata={}, page_con